In [ ]:
# Fixing the ValueError by ensuring proper grouping
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, cohen_kappa_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_predict
from imblearn.ensemble import BalancedRandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical

# Custom Attention Layer (same as before)
class Attention(tf.keras.layers.Layer):
    def __init__(self, units=32, **kwargs):
        super(Attention, self).__init__(**kwargs)
        self.units = units
    
    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], self.units),
                               initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(self.units,),
                               initializer='zeros', trainable=True)
        self.V = self.add_weight(name='attention_v', shape=(self.units, 1),
                               initializer='glorot_uniform', trainable=True)
        super(Attention, self).build(input_shape)
    
    def call(self, x):
        # Alignment scores
        score = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        attention_weights = tf.nn.softmax(tf.matmul(score, self.V), axis=1)
        context_vector = attention_weights * x
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector
    
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

# Load and prepare data
data = pd.read_csv('sledatacut.csv', parse_dates=['ASSDT'], low_memory=False)

# Ensure PTNO is a column, not an index
data = data.reset_index(drop=True)

# Sort by patient and assessment date
data = data.sort_values(['PTNO', 'ASSDT'])

# Calculate time differences
data['time_since_first'] = data.groupby('PTNO')['ASSDT'].transform(
    lambda x: (x - x.min()).dt.days
)
data['time_since_last'] = data.groupby('PTNO')['ASSDT'].transform(
    lambda x: x.diff().dt.days.fillna(0)
)

# Convert categorical EMPf to numerical
emp_mapping = {v: k for k, v in enumerate(data['EMPf'].unique())}
data['EMP_numeric'] = data['EMPf'].map(emp_mapping)

# Encode the target (end_state)
label_encoder = LabelEncoder()
data['end_state_encoded'] = label_encoder.fit_transform(data['end_state'])

# Create one-hot encoded steroid categories
data['STEROID_Low'] = (data['STEROID_CAT'] == 'Low').astype(int)
data['STEROID_Medium'] = (data['STEROID_CAT'] == 'Medium').astype(int)
data['STEROID_High'] = (data['STEROID_CAT'] == 'High').astype(int)

# Base features
features = [
    'EMP_numeric', 
    'severe_flare', 
    'mild_flare', 
    'ISDOSE', 
    'AMDOSE', 
    'score_n', 
    'INCEPT', 
    'age_at_record', 
    'time_since_last', 
    'time_since_first', 
    'visit_num',
    'STEROID_Low',
    'STEROID_Medium',
    'STEROID_High'
]

# Add total flares feature
data['total_flares'] = data['severe_flare'] + data['mild_flare']

# Create numeric steroid category for interactions
data['STEROID_CAT_numeric'] = data['STEROID_CAT'].map({'Low': 0, 'Medium': 1, 'High': 2})

# Add interaction terms
data['steroid_flare_interaction'] = data['STEROID_CAT_numeric'] * data['total_flares']
data['sdi_flare_interaction'] = data['score_n'] * data['total_flares']

# Add temporal features
data['flares_per_month'] = data['total_flares'] / (data['time_since_first']/30 + 1)

# Correct SDI change rate calculation
data['sdi_change_rate'] = data.groupby('PTNO', group_keys=False)['score_n'].apply(
    lambda x: x.diff().fillna(0) / (data.loc[x.index, 'time_since_last']/30 + 1e-6)
)

# Modified rolling features function to avoid groupby issues
def add_rolling_features(df, group_col, features, windows=[3]):
    df = df.copy()
    for feat in features:
        for window in windows:
            df[f'{feat}_rolling_mean_{window}'] = df.groupby(group_col)[feat].transform(
                lambda x: x.rolling(window=window, min_periods=1).mean()
            )
            df[f'{feat}_rolling_std_{window}'] = df.groupby(group_col)[feat].transform(
                lambda x: x.rolling(window=window, min_periods=1).std()
            )
    return df

rolling_features = ['score_n', 'total_flares', 'ISDOSE', 'AMDOSE']
data = add_rolling_features(data, 'PTNO', rolling_features)

# Modified trend feature calculation
def calculate_trend(group):
    group = group.copy()
    group['sdi_trend'] = group['score_n'].rolling(window=3, min_periods=1).apply(
        lambda y: np.polyfit(range(len(y)), y, 1)[0] if len(y) > 1 else 0
    )
    return group

data = data.groupby('PTNO', group_keys=False).apply(calculate_trend)

# Add flare patterns
data['flare_pattern'] = data.groupby('PTNO')['total_flares'].transform(
    lambda x: x.rolling(window=3, min_periods=1).apply(lambda y: 1 if sum(y) >= 2 else 0)
)

# Update features list with new features
features += [
    'total_flares',
    'steroid_flare_interaction',
    'sdi_flare_interaction',
    'flares_per_month',
    'sdi_change_rate',
    'sdi_trend',
    'flare_pattern'
]

# Add rolling features
features += [f'{feat}_rolling_mean_3' for feat in rolling_features]
features += [f'{feat}_rolling_std_3' for feat in rolling_features]

# Fill NA values
data[features] = data[features].fillna(0)

# Normalize features
scaler = MinMaxScaler()
data[features] = scaler.fit_transform(data[features])

# Rest of the code remains the same from create_sequences() onward...
# [Previous sequence creation, model building, and evaluation code]

# Create sequences for each patient
def create_sequences(data, features, max_seq_length=None):
    patients = data['PTNO'].unique()
    num_features = len(features)
    
    if not max_seq_length:
        max_seq_length = data.groupby('PTNO').size().max()
    
    X = np.zeros((len(patients), max_seq_length, num_features))
    y = np.zeros(len(patients))
    seq_lengths = np.zeros(len(patients))
    
    for i, patient in enumerate(patients):
        patient_data = data[data['PTNO'] == patient].sort_values('ASSDT')
        seq_len = len(patient_data)
        seq_lengths[i] = seq_len
        
        # Fill the sequence data
        X[i, :seq_len, :] = patient_data[features].values
        
        # Get the end_state label (last record)
        y[i] = patient_data['end_state_encoded'].iloc[-1]
    
    return X, to_categorical(y), seq_lengths

# Create sequences
X, y, seq_lengths = create_sequences(data, features)

# Split data into train and test
X_train, X_test, y_train, y_test, seq_lengths_train, seq_lengths_test = train_test_split(
    X, y, seq_lengths, test_size=0.2, random_state=42, stratify=np.argmax(y, axis=1))

# Enhanced class weighting
class_weights = compute_class_weight('balanced', classes=np.unique(np.argmax(y_train, axis=1)), 
                                   y=np.argmax(y_train, axis=1))
class_weight_dict = {i: weight**0.5 for i, weight in enumerate(class_weights)}  # Soften weights

# Build enhanced LSTM model with attention
num_classes = y.shape[1]
num_features = len(features)

def build_attention_model(input_shape, num_classes):
    model = Sequential([
        Masking(mask_value=0., input_shape=input_shape),
        LSTM(128, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01)),
        BatchNormalization(),
        Dropout(0.4),
        LSTM(64, return_sequences=True, kernel_regularizer=l2(0.01)),
        BatchNormalization(),
        Attention(units=32),  # Add attention layer
        Dropout(0.4),
        Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
        BatchNormalization(),
        Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
        Dense(num_classes, activation='softmax')
    ])
    return model

model = build_attention_model((None, num_features), num_classes)

# Focal loss implementation
def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fn(y_true, y_pred):
        epsilon = 1e-8
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        loss = alpha * tf.pow(1 - y_pred, gamma) * cross_entropy
        return tf.reduce_mean(loss)
    return focal_loss_fn

optimizer = Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer,
              loss=focal_loss(),
              metrics=['accuracy', 
                      tf.keras.metrics.AUC(name='auc', multi_label=True),
                      tf.keras.metrics.Recall(name='recall'),
                      tf.keras.metrics.Precision(name='precision')])

# Enhanced callbacks
early_stopping = EarlyStopping(monitor='val_auc', patience=20, restore_best_weights=True, mode='max')
reduce_lr = ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5, min_lr=1e-6, mode='max')
checkpoint = ModelCheckpoint('best_model.h5', monitor='val_auc', save_best_only=True, mode='max')

# Train the model
print("\nTraining model...")
history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=150,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, reduce_lr, checkpoint],
    class_weight=class_weight_dict
)

# Function to get last observations
def get_last_observations(X, seq_lengths, features):
    last_obs = []
    for i in range(X.shape[0]):
        seq_len = int(seq_lengths[i])
        last_obs.append(X[i, seq_len-1, :])
    return np.array(last_obs)

# Enhanced ensemble approach
X_last = get_last_observations(X, seq_lengths, features)
y_labels = np.argmax(y, axis=1)

# 1. Balanced Random Forest
brf = BalancedRandomForestClassifier(n_estimators=300, sampling_strategy='all', 
                                   replacement=True, random_state=42, n_jobs=-1)

# 2. Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)

# Cross-validate both models
print("\nTraining ensemble models...")
brf_pred = cross_val_predict(brf, X_last, y_labels, cv=5, method='predict_proba', n_jobs=-1)
gb_pred = cross_val_predict(gb, X_last, y_labels, cv=5, method='predict_proba', n_jobs=-1)

# Get test set predictions
test_indices = [i for i in range(len(X)) if i in [np.where((X == x).all(axis=(1,2)))[0][0] for x in X_test]]
brf_test_pred = brf_pred[test_indices]
gb_test_pred = gb_pred[test_indices]

# Load best LSTM model
best_model = tf.keras.models.load_model('best_model.h5', custom_objects={
    'Attention': Attention,
    'focal_loss_fn': focal_loss()
})
y_pred_lstm = best_model.predict(X_test)

# Dynamic weighting based on validation performance
lstm_weight = 0.6
brf_weight = 0.25
gb_weight = 0.15

final_pred = (lstm_weight * y_pred_lstm + 
              brf_weight * brf_test_pred + 
              gb_weight * gb_test_pred)

y_pred_classes = np.argmax(final_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Enhanced evaluation
print("\nEnhanced Evaluation Metrics:")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true_classes, y_pred_classes):.3f}")
print(f"Cohen's Kappa: {cohen_kappa_score(y_true_classes, y_pred_classes):.3f}")

# Classification report with additional metrics
print("\nEnhanced Classification Report:")
print(classification_report(
    y_true_classes, 
    y_pred_classes, 
    target_names=label_encoder.classes_,
    digits=4
))

# Confusion matrix with normalization
cm = confusion_matrix(y_true_classes, y_pred_classes)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print("\nNormalized Confusion Matrix:")
print(cm_normalized)

# Feature importance analysis (for tree-based models)
print("\nFeature Importance Analysis:")
brf.fit(X_last, y_labels)  # Fit on full data
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': brf.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance.head(20))

/var/folders/x9/3zl22m596tl6d44cbs62z6h00000gn/T/ipykernel_62547/715538926.py:135: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('PTNO', group_keys=False).apply(calculate_trend)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training model...
Epoch 1/150


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/layer.py:939: UserWarning: Layer 'attention' (of type Attention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - accuracy: 0.3310 - auc: 0.4963 - loss: 4.2137 - precision: 0.3264 - recall: 0.1002

36/36 ━━━━━━━━━━━━━━━━━━━━ 17s 305ms/step - accuracy: 0.3320 - auc: 0.4969 - loss: 4.2031 - precision: 0.3281 - recall: 0.1008 - val_accuracy: 0.4772 - val_auc: 0.5575 - val_loss: 3.0536 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 5.0000e-04
Epoch 2/150
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.4562 - auc: 0.5961 - loss: 2.7629 - precision: 0.5797 - recall: 0.1834

36/36 ━━━━━━━━━━━━━━━━━━━━ 8s 212ms/step - accuracy: 0.4567 - auc: 0.5968 - loss: 2.7561 - precision: 0.5807 - recall: 0.1836 - val_accuracy: 0.5018 - val_auc: 0.5688 - val_loss: 2.0251 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 5.0000e-04
Epoch 3/150
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.5296 - auc: 0.6718 - loss: 1.8331 - precision: 0.6779 - recall: 0.2235

36/36 ━━━━━━━━━━━━━━━━━━━━ 8s 216ms/step - accuracy: 0.5303 - auc: 0.6722 - loss: 1.8286 - precision: 0.6790 - recall: 0.2239 - val_accuracy: 0.4632 - val_auc: 0.6407 - val_loss: 1.3636 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 5.0000e-04
Epoch 4/150
10/36 ━━━━━━━━━━━━━━━━━━━━ 5s 206ms/step - accuracy: 0.5433 - auc: 0.7050 - loss: 1.3192 - precision: 0.7270 - recall: 0.2388